# TSP Variant Comparison

Loads all per-variant result files from `results/` and visualises
tour quality (optimality gap %) and cost across instances and variants.

Run benchmarks first:
```bash
python -m benchmark.runner --all
```

In [ ]:
import json
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Make sure project root is on the path
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

RESULTS_DIR = os.path.join(ROOT, "results")
print(f"Loading results from: {RESULTS_DIR}")

In [ ]:
# ── Load all result files ──────────────────────────────────────────────────

def load_results(results_dir: str) -> dict[str, dict]:
    """Return {variant_name: result_dict} for every .json in results/."""
    out = {}
    if not os.path.isdir(results_dir):
        print("results/ directory not found — run a benchmark first.")
        return out
    for fname in sorted(os.listdir(results_dir)):
        if fname.endswith(".json"):
            with open(os.path.join(results_dir, fname)) as f:
                data = json.load(f)
            out[data["variant"]] = data
    return out


results = load_results(RESULTS_DIR)
print(f"Variants loaded: {list(results.keys())}")

In [ ]:
# ── Build tidy DataFrames ──────────────────────────────────────────────────

rows = []
for variant, data in results.items():
    for instance, inst in data["instances"].items():
        rows.append({
            "variant":      variant,
            "instance":     instance,
            "cost":         inst["cost"],
            "optimal_cost": inst["optimal_cost"],
            "gap_pct":      inst["gap_pct"],
            "time_sec":     inst["time_sec"],
            "n_nodes":      inst["n_nodes"],
        })

df = pd.DataFrame(rows)

# Summary per variant
summary = (
    df.groupby("variant")
    .agg(
        avg_gap_pct=("gap_pct", "mean"),
        max_gap_pct=("gap_pct", "max"),
        total_time_sec=("time_sec", "sum"),
    )
    .sort_values("avg_gap_pct")
)

print("Summary (lower gap = better):")
print(summary.to_string(float_format="{:.3f}".format))

In [ ]:
# ── Bar chart: average gap% per variant ───────────────────────────────────

fig, ax = plt.subplots(figsize=(max(6, len(results) * 1.4), 5))

variants_sorted = summary.index.tolist()
avg_gaps = summary["avg_gap_pct"].values
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(variants_sorted)))

bars = ax.bar(variants_sorted, avg_gaps, color=colors, edgecolor="white", linewidth=0.8)

for bar, val in zip(bars, avg_gaps):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{val:.2f}%",
        ha="center", va="bottom", fontsize=9,
    )

ax.set_ylabel("Average optimality gap (%)")
ax.set_title("Average gap vs. known optimum — all test instances")
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
ax.grid(axis="y", which="major", linestyle="--", alpha=0.4)
ax.set_ylim(0, max(avg_gaps) * 1.25 + 0.5)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: gap% per variant × instance ─────────────────────────────────

from config.test_suite import TEST_INSTANCES

pivot = df.pivot(index="instance", columns="variant", values="gap_pct")
# Reorder rows to match test-suite ordering
ordered_instances = [i for i in TEST_INSTANCES if i in pivot.index]
pivot = pivot.loc[ordered_instances]

fig, ax = plt.subplots(figsize=(max(6, len(results) * 1.6), max(5, len(ordered_instances) * 0.55)))

im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn_r", vmin=0)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=25, ha="right", fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)

for r in range(pivot.shape[0]):
    for c in range(pivot.shape[1]):
        val = pivot.values[r, c]
        if not np.isnan(val):
            ax.text(c, r, f"{val:.1f}%", ha="center", va="center",
                    fontsize=7.5, color="black")

plt.colorbar(im, ax=ax, label="Optimality gap (%)")
ax.set_title("Optimality gap (%) — variant × instance")
plt.tight_layout()
plt.show()

In [ ]:
# ── Line chart: gap% vs. instance size ────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 5))

for variant in variants_sorted:
    subset = df[df["variant"] == variant].copy()
    subset = subset.sort_values("n_nodes")
    subset_with_opt = subset.dropna(subset=["gap_pct"])
    ax.plot(
        subset_with_opt["n_nodes"],
        subset_with_opt["gap_pct"],
        marker="o", label=variant, linewidth=1.5,
    )

ax.set_xlabel("Problem size (nodes)")
ax.set_ylabel("Optimality gap (%)")
ax.set_title("Gap vs. problem size — how does quality scale?")
ax.legend()
ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Raw numbers table ─────────────────────────────────────────────────────

table = df.pivot_table(
    index="instance",
    columns="variant",
    values="gap_pct",
    aggfunc="first",
)
table.loc["AVERAGE"] = table.mean()
table.loc[ordered_instances] if ordered_instances else table

display(table.style
    .format("{:.2f}%", na_rep="—")
    .background_gradient(cmap="RdYlGn_r", axis=None)
    .set_caption("Optimality gap (%) by variant and instance")
)